RAG Fusion

In [53]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [54]:
from langchain_community.vectorstores import FAISS

Chunking

In [55]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

LLM and Embedding model

In [56]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0
)

In [57]:
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

In [58]:
loader = PyPDFLoader("attention.pdf")
chunks = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
docs = text_splitter.split_documents(chunks)

In [59]:
db = FAISS.from_documents(docs, embeddings)

In [60]:
retriever = db.as_retriever(k=5)

Multi-Query Generation

In [61]:
from langchain_core.prompts import ChatPromptTemplate

In [62]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [63]:
prompt = """You are a helpful assistant that generates multiple search queries based on a single input query. \n

Generate multiple search queries related to: {question} \n

Output (4 queries):"""

In [64]:
multi_prompt = ChatPromptTemplate.from_template(prompt)

In [65]:
generate_queries = (
    multi_prompt
    | llm
    | StrOutputParser()
    | RunnableLambda(lambda x: x.split("\n"))
)

Re-Ranking

In [66]:
def reciprocal_rank_fusion(results, k=60):

    fused_scores = {}
    doc_map = {}

    for docs in results:
        for rank, doc in enumerate(docs):

            doc_id = doc.page_content

            if doc_id not in fused_scores:
                fused_scores[doc_id] = 0
                doc_map[doc_id] = doc

            fused_scores[doc_id] += 1 / (rank + k) # reciprocal rank fusion formula

    reranked = sorted(
        fused_scores.keys(),
        key=lambda x: fused_scores[x],
        reverse=True
    )

    return [doc_map[x] for x in reranked]

Retrieval Chain

In [67]:
fusion_chain = (
    generate_queries
    | retriever.map()
    | RunnableLambda(reciprocal_rank_fusion)
)

Final Generation Chain

In [68]:
answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context.

Context:
{context}

Question:
{question}
""")

rag_fusion_chain = (
    {
        "context": fusion_chain,
        "question": RunnablePassthrough()
    }
    | answer_prompt
    | llm
    | StrOutputParser()
)

In [70]:
response = rag_fusion_chain.invoke(
    "What is attention?"
)

print(response)

Based on the provided context, attention (specifically multi-head attention and self-attention) is a mechanism with the following characteristics:

*   **Core Function:** It allows a model to "jointly attend to information from different representation subspaces at different positions."
*   **Sequence Mapping:** Self-attention layers are used to map one variable-length sequence of symbol representations to another sequence of equal length.
*   **Long-Distance Dependencies:** The mechanism is capable of following long-distance dependencies within a network, such as attending to a distant dependency of a verb to complete a phrase (e.g., "making...more difficult").
*   **Structure and Interpretation:** Attention distributions often exhibit behavior related to the syntactic and semantic structure of sentences. Individual "attention heads" can learn to perform different tasks, such as anaphora resolution.
*   **Multi-Head Structure:** In the described model, multi-head attention employs mul